# Lab 1 - Simulating the Counterfactual World

*SDAIA Academy · Experimentation and Causal Inference · STARTER notebook*

## Objective
Build a synthetic population where **you** control the full potential-outcomes table, then experience why the naive estimate misleads and how randomization fixes it.

Deliverable: true ATE, naive (selection-biased) estimate, randomized estimate with a design-based SE, a balance plot, a randomization-inference p-value, and a sample-size sweep showing that bias does not shrink with n.

Context: Injaz users and a **guided document uploader**. `X` is digital literacy, a confounder: it raises baseline completion *and* makes a user more likely to opt into the new uploader.

In [ ]:
import sys; sys.path.insert(0, '..')   # so `causal_utils` is importable from starter/ or solution/
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from causal_utils import *
plt.rcParams['figure.figsize'] = (7, 3.5)
rng = np.random.default_rng(213)
DATA = '../data'

## Step 1 - Generate N = 5,000 units with a confounder X and both potential outcomes
Complete `simulate_population` so that `p1 = p0 + true_effect` and both `y0` and `y1` exist for every unit.

In [ ]:
def simulate_population(n=5_000, true_effect=0.05, seed=213):
    rng = np.random.default_rng(seed)
    x = rng.beta(2, 2, size=n)                          # digital literacy in [0,1]
    age = rng.integers(18, 70, size=n)
    p0 = np.clip(0.35 + 0.40 * x - 0.002 * (age - 40), 0.01, 0.99)   # completion prob WITHOUT uploader
    p1 = ...                                            # TODO: add true_effect and clip to [0.01, 0.99]
    y0 = rng.binomial(1, p0)
    y1 = ...                                            # TODO: draw Y(1)
    df = pd.DataFrame({'x': x, 'age': age, 'p0': p0, 'p1': p1, 'y0': y0, 'y1': y1})
    df['tau_i'] = ...                                   # TODO: individual treatment effect
    return df

pop = simulate_population()
pop.head()

## Step 2 - The oracle ATE
Only possible because the simulator exposes both columns. Compare the probability-scale ATE with the realised mean of `tau_i`.

In [ ]:
true_ate = ...        # TODO: mean of p1 - p0
realised_ate = ...    # TODO: mean of tau_i
print(f"True ATE (planted): {true_ate:.4f}   realised: {realised_ate:.4f}")

## Step 3 - Induce selection: P(T=1) increases with X, then compute the naive difference
The observed outcome is `Y = T*Y(1) + (1-T)*Y(0)`. The naive estimator is `mean(Y | T=1) - mean(Y | T=0)`.

In [ ]:
def observed(df, t):
    return ...            # TODO: reveal y1 where t==1 else y0 (np.where)

def naive_diff(df, t):
    y = observed(df, t)
    return ...            # TODO: mean(y|t==1) - mean(y|t==0)

p_optin = 0.15 + 0.70 * pop['x']                 # literate users opt in more -> confounded
pop['T_self'] = rng.binomial(1, p_optin)
naive_self = naive_diff(pop, pop['T_self'].values)
print(f"Naive estimate under self-selection: {naive_self:.4f}   (truth {true_ate:.4f})")

## Step 4 - Re-assign T completely at random and recompute
Same units, same potential outcomes. Only the assignment mechanism changes. Report the design-based (Neyman) standard error.

In [ ]:
pop['T_rand'] = rng.binomial(1, 0.5, size=len(pop))
y_rand = observed(pop, pop['T_rand'].values)
res = diff_in_means(y_rand, pop['T_rand'].values)     # from causal_utils: Neyman estimator + conservative SE
print(f"Randomized estimate: {res['estimate']:.4f}  SE={res['se']:.4f}  95% CI [{res['ci_low']:.4f}, {res['ci_high']:.4f}]")

## Step 5 - Decompose the naive difference and plot covariate balance
`naive = ATT + selection bias`, where bias = `E[Y(0)|T=1] - E[Y(0)|T=0]`. We can compute the bias term directly because Y(0) is known for everyone.

In [ ]:
for t_col, label in [('T_self', 'self-selection'), ('T_rand', 'randomized')]:
    t = pop[t_col].values
    att = ...     # TODO: mean tau_i among treated
    bias = ...    # TODO: mean y0 among treated minus mean y0 among control
    print(f"{label:15s} ATT={att:+.4f}  selection bias={bias:+.4f}  ATT+bias={att+bias:+.4f}  naive={naive_diff(pop, t):+.4f}")

# TODO: plot histograms of x by arm for T_self and T_rand; print smd(pop, 'x', t_col) in the title

## Step 6 - Randomization-inference p-value (1,000 label permutations)
Fisher's sharp null: no effect for *any* unit. Permute the treatment labels the way you assigned them and count how often the permuted difference is at least as extreme as the observed one.

In [ ]:
# TODO: call randomization_test(y_rand, pop['T_rand'].values, n_perm=1000) and plot the null draws
ri = ...
print(ri['p_value'])

## Step 7 - Bias does not shrink with n
Sweep n over {1k, 10k, 100k, 1M}. The self-selection estimate converges to the *biased* value; only the randomized estimate converges to the truth.

In [ ]:
# TODO: for n in [1_000, 10_000, 100_000, 1_000_000], simulate, assign both ways, store naive_diff for each; plot vs n (log x)

## Step 8 - Three-sentence conclusion for a non-technical manager
Write it in the cell below. It must mention: what the naive number measures, why more data would not have helped, and what design would.

*Your answer:*

...

### Fast finishers
Regress `y` on `T_self` and `x` (`smf.ols('y ~ T + x')`). Does covariate adjustment remove the bias? Partly? Why (think about `age`)? This previews Day 3.